In [1]:
# ASSAY controlled transaction-fraud benchmark
# This benchmark validates training mechanics only. It is never merged with MCA/NSE labels.
import json
import os
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd

ASSAY_SEED = 106
BENCHMARK_REPOSITORY = "https://github.com/Fraud-Detection-Handbook/simulated-data-transformed.git"
BENCHMARK_DIRECTORY = Path("/content/simulated-data-transformed")

os.environ["PYTHONHASHSEED"] = str(ASSAY_SEED)
random.seed(ASSAY_SEED)
np.random.seed(ASSAY_SEED)

if not BENCHMARK_DIRECTORY.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", BENCHMARK_REPOSITORY, str(BENCHMARK_DIRECTORY)],
        check=True,
    )

benchmark_revision = subprocess.check_output(
    ["git", "-C", str(BENCHMARK_DIRECTORY), "rev-parse", "HEAD"],
    text=True,
).strip()
benchmark_files = sorted((BENCHMARK_DIRECTORY / "data").glob("*.pkl"))

print(json.dumps({
    "benchmark": "fraud_detection_handbook_simulated_transactions",
    "benchmark_revision": benchmark_revision,
    "daily_files": len(benchmark_files),
    "seed": ASSAY_SEED,
    "label_boundary": "transaction_fraud_only_not_mca_entity_risk",
}, indent=2))


{
  "benchmark": "fraud_detection_handbook_simulated_transactions",
  "benchmark_revision": "6e3ca5849b4681430388056d3f1dcfb41d4e8269",
  "daily_files": 183,
  "seed": 106,
  "label_boundary": "transaction_fraud_only_not_mca_entity_risk"
}


In [2]:
# Freeze non-random temporal splits before model fitting.
# Seven-day embargoes reduce near-boundary leakage from rolling aggregates.
TRAIN_FILES = benchmark_files[:90]
TRAIN_EMBARGO_FILES = benchmark_files[90:97]
VALIDATION_FILES = benchmark_files[97:127]
VALIDATION_EMBARGO_FILES = benchmark_files[127:134]
TEST_FILES = benchmark_files[134:]

def audit_daily_file(file_path):
    daily_frame = pd.read_pickle(file_path)
    return {
        "file": file_path.name,
        "rows": len(daily_frame),
        "fraud_rows": int(daily_frame["TX_FRAUD"].sum()),
        "fraud_rate": float(daily_frame["TX_FRAUD"].mean()),
    }

daily_audit = pd.DataFrame([audit_daily_file(path) for path in benchmark_files])
first_frame = pd.read_pickle(benchmark_files[0])

def split_summary(split_name, split_files):
    split_audit = daily_audit[daily_audit["file"].isin([path.name for path in split_files])]
    rows = int(split_audit["rows"].sum())
    fraud_rows = int(split_audit["fraud_rows"].sum())
    return {
        "split": split_name,
        "first_file": split_files[0].name,
        "last_file": split_files[-1].name,
        "files": len(split_files),
        "rows": rows,
        "fraud_rows": fraud_rows,
        "fraud_rate": fraud_rows / rows,
    }

split_audit = [
    split_summary("development_train", TRAIN_FILES),
    split_summary("development_validation", VALIDATION_FILES),
    split_summary("temporal_test", TEST_FILES),
]
print(json.dumps({
    "columns": list(first_frame.columns),
    "splits": split_audit,
    "train_embargo": [TRAIN_EMBARGO_FILES[0].name, TRAIN_EMBARGO_FILES[-1].name],
    "validation_embargo": [VALIDATION_EMBARGO_FILES[0].name, VALIDATION_EMBARGO_FILES[-1].name],
}, indent=2))


{
  "columns": [
    "TRANSACTION_ID",
    "TX_DATETIME",
    "CUSTOMER_ID",
    "TERMINAL_ID",
    "TX_AMOUNT",
    "TX_TIME_SECONDS",
    "TX_TIME_DAYS",
    "TX_FRAUD",
    "TX_FRAUD_SCENARIO",
    "TX_DURING_WEEKEND",
    "TX_DURING_NIGHT",
    "CUSTOMER_ID_NB_TX_1DAY_WINDOW",
    "CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW",
    "CUSTOMER_ID_NB_TX_7DAY_WINDOW",
    "CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW",
    "CUSTOMER_ID_NB_TX_30DAY_WINDOW",
    "CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW",
    "TERMINAL_ID_NB_TX_1DAY_WINDOW",
    "TERMINAL_ID_RISK_1DAY_WINDOW",
    "TERMINAL_ID_NB_TX_7DAY_WINDOW",
    "TERMINAL_ID_RISK_7DAY_WINDOW",
    "TERMINAL_ID_NB_TX_30DAY_WINDOW",
    "TERMINAL_ID_RISK_30DAY_WINDOW"
  ],
  "splits": [
    {
      "split": "development_train",
      "first_file": "2018-04-01.pkl",
      "last_file": "2018-06-29.pkl",
      "files": 90,
      "rows": 863006,
      "fraud_rows": 6761,
      "fraud_rate": 0.007834244489609574
    },
    {
      "split": "development_validation"

In [3]:
# GPU candidate model on the controlled transaction benchmark.
# Model selection uses PR-AUC and fixed review capacity, never raw accuracy.
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

EXCLUDED_COLUMNS = {
    "TRANSACTION_ID", "TX_DATETIME", "CUSTOMER_ID", "TERMINAL_ID",
    "TX_TIME_SECONDS", "TX_TIME_DAYS", "TX_FRAUD", "TX_FRAUD_SCENARIO",
}
FEATURE_COLUMNS = [column for column in first_frame.columns if column not in EXCLUDED_COLUMNS]
TARGET_COLUMN = "TX_FRAUD"
REVIEW_CAPACITY = 0.01

def load_split(split_files):
    return pd.concat(
        [pd.read_pickle(path)[FEATURE_COLUMNS + [TARGET_COLUMN]] for path in split_files],
        ignore_index=True,
    )

def fixed_capacity_metrics(target, score, review_capacity=REVIEW_CAPACITY):
    target_array = np.asarray(target, dtype=np.int8)
    score_array = np.asarray(score, dtype=np.float64)
    review_rows = max(1, int(np.ceil(len(target_array) * review_capacity)))
    selected_indices = np.argpartition(score_array, -review_rows)[-review_rows:]
    selected_targets = target_array[selected_indices]
    base_rate = float(target_array.mean())
    precision = float(selected_targets.mean())
    recall = float(selected_targets.sum() / target_array.sum())
    return {
        "rows": len(target_array),
        "positive_rows": int(target_array.sum()),
        "base_rate": base_rate,
        "pr_auc": float(average_precision_score(target_array, score_array)),
        "roc_auc": float(roc_auc_score(target_array, score_array)),
        "brier_score": float(brier_score_loss(target_array, score_array)),
        "accuracy_diagnostic_only": float(((score_array >= 0.5) == target_array).mean()),
        "review_capacity": review_capacity,
        "reviewed_rows": review_rows,
        "precision_at_capacity": precision,
        "recall_at_capacity": recall,
        "lift_at_capacity": precision / base_rate,
    }

train_frame = load_split(TRAIN_FILES)
validation_frame = load_split(VALIDATION_FILES)
test_frame = load_split(TEST_FILES)

X_train = train_frame[FEATURE_COLUMNS].astype("float32")
y_train = train_frame[TARGET_COLUMN].astype("int8")
X_validation = validation_frame[FEATURE_COLUMNS].astype("float32")
y_validation = validation_frame[TARGET_COLUMN].astype("int8")
X_test = test_frame[FEATURE_COLUMNS].astype("float32")
y_test = test_frame[TARGET_COLUMN].astype("int8")

class_ratio = float((len(y_train) - y_train.sum()) / y_train.sum())
model_started_at = time.time()
transaction_risk_model = XGBClassifier(
    n_estimators=1200,
    max_depth=8,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    scale_pos_weight=float(np.sqrt(class_ratio)),
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    device="cuda",
    random_state=ASSAY_SEED,
    early_stopping_rounds=75,
)
transaction_risk_model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    verbose=100,
)
training_wall_seconds = time.time() - model_started_at

validation_scores = transaction_risk_model.predict_proba(X_validation)[:, 1]
test_scores = transaction_risk_model.predict_proba(X_test)[:, 1]
amount_only_test_scores = test_frame["TX_AMOUNT"].rank(pct=True).to_numpy()

benchmark_metrics = {
    "benchmark_revision": benchmark_revision,
    "seed": ASSAY_SEED,
    "features": FEATURE_COLUMNS,
    "best_iteration": int(transaction_risk_model.best_iteration),
    "training_wall_seconds": training_wall_seconds,
    "scale_pos_weight": float(np.sqrt(class_ratio)),
    "validation": fixed_capacity_metrics(y_validation, validation_scores),
    "temporal_test": fixed_capacity_metrics(y_test, test_scores),
    "amount_only_temporal_test_baseline": fixed_capacity_metrics(y_test, amount_only_test_scores),
    "claim_boundary": "controlled_simulated_transaction_fraud_only",
}
print(json.dumps(benchmark_metrics, indent=2))


[0]	validation_0-aucpr:0.60125
[100]	validation_0-aucpr:0.69634
[200]	validation_0-aucpr:0.70471
[300]	validation_0-aucpr:0.70617
[395]	validation_0-aucpr:0.70607


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [14:30:18] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


{
  "benchmark_revision": "6e3ca5849b4681430388056d3f1dcfb41d4e8269",
  "seed": 106,
  "features": [
    "TX_AMOUNT",
    "TX_DURING_WEEKEND",
    "TX_DURING_NIGHT",
    "CUSTOMER_ID_NB_TX_1DAY_WINDOW",
    "CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW",
    "CUSTOMER_ID_NB_TX_7DAY_WINDOW",
    "CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW",
    "CUSTOMER_ID_NB_TX_30DAY_WINDOW",
    "CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW",
    "TERMINAL_ID_NB_TX_1DAY_WINDOW",
    "TERMINAL_ID_RISK_1DAY_WINDOW",
    "TERMINAL_ID_NB_TX_7DAY_WINDOW",
    "TERMINAL_ID_RISK_7DAY_WINDOW",
    "TERMINAL_ID_NB_TX_30DAY_WINDOW",
    "TERMINAL_ID_RISK_30DAY_WINDOW"
  ],
  "best_iteration": 320,
  "training_wall_seconds": 8.6441490650177,
  "scale_pos_weight": 11.25365394489159,
  "validation": {
    "rows": 286826,
    "positive_rows": 2559,
    "base_rate": 0.008921785333268253,
    "pr_auc": 0.7062418503145635,
    "roc_auc": 0.9001315110500369,
    "brier_score": 0.005693321303594192,
    "accuracy_diagnostic_only": 0.993442017111

In [4]:
# Persist model evidence with versions and checksums.
import hashlib
import platform
import shutil

import sklearn
import xgboost
from google.colab import files

ARTIFACT_DIRECTORY = Path("/content/assay-controlled-benchmark")
ARTIFACT_DIRECTORY.mkdir(exist_ok=True)
model_path = ARTIFACT_DIRECTORY / "transaction_fraud_xgboost.json"
metrics_path = ARTIFACT_DIRECTORY / "metrics.json"
manifest_path = ARTIFACT_DIRECTORY / "manifest.json"
transaction_risk_model.save_model(model_path)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as artifact_file:
        for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()
benchmark_metrics["runtime"] = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "gpu": gpu_name,
}
metrics_path.write_text(json.dumps(benchmark_metrics, indent=2, sort_keys=True))
manifest = {
    "benchmark_revision": benchmark_revision,
    "model_sha256": file_sha256(model_path),
    "metrics_sha256": file_sha256(metrics_path),
    "claim_boundary": "controlled_simulated_transaction_fraud_only",
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
archive_path = shutil.make_archive(
    f"/content/assay-controlled-benchmark-{benchmark_revision[:12]}",
    "zip",
    ARTIFACT_DIRECTORY,
)
print(json.dumps({**manifest, "archive_path": archive_path, "archive_sha256": file_sha256(archive_path)}, indent=2))
files.download(archive_path)


{
  "benchmark_revision": "6e3ca5849b4681430388056d3f1dcfb41d4e8269",
  "model_sha256": "fc79b3263dc192abe3458d0ef0371858d7fc72461708fb5db56a6ff7cb660624",
  "metrics_sha256": "f8df02cfd4350e6661cf2b124c0647f59939d2413975362507001c39f5e63a6b",
  "claim_boundary": "controlled_simulated_transaction_fraud_only",
  "archive_path": "/content/assay-controlled-benchmark-6e3ca5849b46.zip",
  "archive_sha256": "fa9cd8fab8978ae66a355ae89fc723e58cba822c93b39756302e4fbfd5449309"
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>